# Chapter 15 Companion Notebook: Vision Models in Business Analytics

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
**Notebook author:** Hyunhwan Aiden Lee  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch15_Vision_Models.ipynb)

This notebook accompanies Chapter 15 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




Classroom note: this notebook uses synthetic product images, shelf scenes, and document layouts so students can practice the full workflow in Colab without a paid API or an external dataset. The images are intentionally simple, but the evaluation and governance logic transfers directly to real business vision projects.

Copyright 2026 to present.

## How to use this notebook

Run the cells from top to bottom. Treat the markdown sections as short lecture notes and the code sections as live demonstrations. Keep `FAST_MODE = True` for a classroom run. The optional vision-language foundation model demonstration is off by default because it downloads a public pretrained model.

## Why this matters (business framing)

Computer vision converts visible conditions into measurable signals. A retailer can measure display compliance, a manufacturer can route suspected defects, a marketing team can organize creative assets, and an e-commerce platform can support visual search. The model is only one component. A reliable vision workflow also requires a clear unit of analysis, representative operating conditions, defensible labels, leakage-safe splits, error-cost-aware thresholds, and monitoring after deployment.

This notebook follows the practical logic of Chapter 15. We will build a small business image system from the decision contract outward. Classification and transfer learning form the modeling core. Detection, segmentation, document vision, embeddings, self-supervision, vision transformers, and foundation models are introduced at the level needed to understand their outputs and evaluate whether they create business value.

## Agenda

1. Setup and reproducibility
2. Synthetic business image assets and metadata
3. Vision task families and output contracts
4. Leakage-safe splits and camera-shortcut diagnostics
5. CNN classification and staged transfer learning
6. Calibration, error costs, human review, and slice evaluation
7. Detection and segmentation outputs
8. Document vision and field-level accuracy
9. Vision embeddings for search and recommendation
10. Self-supervision, patch tokens, and foundation model strategy
11. Monitoring, governance, and saved artifacts
12. Exercises

## Learning objectives (measurable)

By the end of this notebook, you should be able to map a business question to an appropriate vision output, construct a group-based split that protects against camera leakage, train a compact CNN, freeze and fine-tune a transferred backbone, calibrate probabilities, choose thresholds using error costs and review capacity, calculate IoU and Dice, extract a document field using layout information, evaluate embedding retrieval with ranking metrics, explain how an image becomes a sequence of transformer patches, and create a monitoring and governance record for a production vision system.

## Connection map

Chapter 14 introduced the deep learning training workflow, including layered transformations, losses, backpropagation, and overfitting controls. This notebook applies those ideas to pixels and spatial outputs. Chapter 16 explains attention mechanically. Here, attention appears only where it helps connect image patches to vision transformers and vision-language foundation models.

The recurring workflow is: define the decision, choose the simplest sufficient output, build representative data and labels, split by the real deployment boundary, establish a baseline, adapt a pretrained representation, evaluate by cost and slice, and monitor the resulting system as a probabilistic sensor.

In [ ]:
# ============================================================
# 1. Setup and reproducibility
# - install missing packages if needed
# - import libraries
# - set seeds and runtime controls
# - configure output folders
# ============================================================
import os
import sys
import json
import math
import time
import copy
import random
import warnings
import importlib
import subprocess
from pathlib import Path


def ensure(pkg_import_name, pip_name=None):
    """Install a package only if it is missing."""
    try:
        importlib.import_module(pkg_import_name)
    except Exception:
        pip_target = pip_name or pkg_import_name
        print(f"Installing {pip_target} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_target])


for import_name, pip_name in [
    ("numpy", None),
    ("pandas", None),
    ("sklearn", "scikit-learn"),
    ("matplotlib", None),
    ("PIL", "Pillow"),
    ("scipy", None),
    ("torch", None),
]:
    ensure(import_name, pip_name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter
from IPython.display import display, Markdown

from scipy.ndimage import binary_dilation, binary_erosion

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 30)

SEED = 685
FAST_MODE = True
RUN_OPTIONAL_CLIP_DEMO = False
LABELED_TRAIN_FRACTION = 0.35

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(1)

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
OUT_DIR = BASE_DIR / "baai_ch15_vision_outputs"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 48
BATCH_SIZE = 128 if DEVICE == "cpu" else 256

if FAST_MODE:
    N_SOURCE = 800
    N_TARGET = 900
    N_CHALLENGE = 240
    PRETRAIN_EPOCHS = 4
    SCRATCH_EPOCHS = 5
    FROZEN_EPOCHS = 3
    FINETUNE_EPOCHS = 8
else:
    N_SOURCE = 1400
    N_TARGET = 1500
    N_CHALLENGE = 400
    PRETRAIN_EPOCHS = 6
    SCRATCH_EPOCHS = 7
    FROZEN_EPOCHS = 4
    FINETUNE_EPOCHS = 10

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print({
    "FAST_MODE": FAST_MODE,
    "N_SOURCE": N_SOURCE,
    "N_TARGET": N_TARGET,
    "N_CHALLENGE": N_CHALLENGE,
    "BATCH_SIZE": BATCH_SIZE,
    "RUN_OPTIONAL_CLIP_DEMO": RUN_OPTIONAL_CLIP_DEMO,
    "LABELED_TRAIN_FRACTION": LABELED_TRAIN_FRACTION,
    "OUT_DIR": str(OUT_DIR),
})

## Utility functions

The helpers below keep later sections focused on modeling and decision design. They calculate classification metrics, choose thresholds, display reliability, and summarize training histories.

In [ ]:
# ============================================================
# Utility functions
# ============================================================

def print_section(title):
    print("=" * len(title))
    print(title)
    print("=" * len(title))


def sigmoid_np(x):
    x = np.asarray(x, dtype=float)
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))


def safe_auc(y_true, prob):
    try:
        return roc_auc_score(y_true, prob)
    except Exception:
        return np.nan


def safe_average_precision(y_true, prob):
    try:
        return average_precision_score(y_true, prob)
    except Exception:
        return np.nan


def expected_calibration_error(y_true, prob, n_bins=10):
    y_true = np.asarray(y_true)
    prob = np.asarray(prob)
    edges = np.linspace(0, 1, n_bins + 1)
    total = len(y_true)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        if hi == 1:
            mask = (prob >= lo) & (prob <= hi)
        else:
            mask = (prob >= lo) & (prob < hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / total) * abs(prob[mask].mean() - y_true[mask].mean())
    return float(ece)


def binary_metrics(y_true, prob, threshold=0.50):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": safe_auc(y_true, prob),
        "average_precision": safe_average_precision(y_true, prob),
        "brier": brier_score_loss(y_true, prob),
        "ece": expected_calibration_error(y_true, prob),
        "pred_positive_rate": float(pred.mean()),
    }


def find_best_f1_threshold(y_true, prob):
    thresholds = np.linspace(0.05, 0.95, 91)
    scores = [f1_score(y_true, prob >= th, zero_division=0) for th in thresholds]
    best_idx = int(np.argmax(scores))
    return float(thresholds[best_idx]), float(scores[best_idx])


def find_min_cost_threshold(y_true, prob, false_negative_cost=8.0, false_positive_cost=1.0):
    rows = []
    for th in np.linspace(0.01, 0.99, 99):
        pred = (prob >= th).astype(int)
        fn = int(((y_true == 1) & (pred == 0)).sum())
        fp = int(((y_true == 0) & (pred == 1)).sum())
        total_cost = false_negative_cost * fn + false_positive_cost * fp
        rows.append({"threshold": th, "false_negatives": fn, "false_positives": fp, "total_cost": total_cost})
    cost_df = pd.DataFrame(rows)
    best_row = cost_df.loc[cost_df["total_cost"].idxmin()]
    return float(best_row["threshold"]), cost_df


def plot_confusion_matrix(y_true, prob, threshold=0.50, title="Confusion matrix"):
    pred = (np.asarray(prob) >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(cm)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["acceptable", "defect"])
    ax.set_yticklabels(["acceptable", "defect"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, int(val), ha="center", va="center")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


def plot_reliability(y_true, probability_sets, title="Reliability diagram"):
    plt.figure(figsize=(6.5, 5))
    plt.plot([0, 1], [0, 1], linestyle="--", label="perfect calibration")
    for label, prob in probability_sets.items():
        frac_pos, mean_pred = calibration_curve(y_true, prob, n_bins=8, strategy="quantile")
        plt.plot(mean_pred, frac_pos, marker="o", label=label)
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed defect rate")
    plt.title(title)
    plt.legend()
    plt.show()


def plot_training_history(history_df, score_col, title):
    plt.figure(figsize=(7, 4))
    plt.plot(history_df["epoch"], history_df["train_loss"], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Training loss")
    plt.title(title + " loss")
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(history_df["epoch"], history_df[score_col], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel(score_col)
    plt.ylim(0, 1.05)
    plt.title(title + " validation score")
    plt.show()


print("Utility helpers ready.")

## 2. Synthetic business image assets and metadata

The simulated setting is a quality-control workflow for packaged products. Each image contains one product, a camera-specific border and background pattern, operating conditions such as lighting, and a label indicating whether the package has a visible defect. The generator also knows the product mask and defect mask, which lets us demonstrate classification, detection, segmentation, and embeddings without external files.

The camera context is deliberately correlated with the defect rate. This creates a realistic shortcut risk. A random split can look strong because the same cameras appear in training and testing, even when the model learns camera context instead of the defect itself.

In [ ]:
# ============================================================
# 2.1 Synthetic product image generator
# ============================================================
FAMILIES = ["beverage", "snack", "cleaner"]
LIGHTINGS = ["normal", "dim", "warm", "glare"]

CAMERA_COLORS = [
    (185, 70, 70),
    (65, 115, 195),
    (70, 165, 100),
    (195, 145, 55),
    (135, 85, 185),
    (65, 165, 175),
    (125, 125, 125),
]

BODY_COLORS = [
    (55, 130, 220),
    (235, 110, 55),
    (65, 180, 110),
    (175, 95, 205),
    (235, 185, 60),
    (65, 175, 195),
]


def _draw_product_shape(draw, mask_draw, family, fill, outline, jitter_x, jitter_y):
    """Draw one package and return its outer box and label box."""
    if family == "beverage":
        x0, x1, y0, y1 = 17 + jitter_x, 31 + jitter_x, 7 + jitter_y, 41 + jitter_y
        draw.rounded_rectangle((x0, y0, x1, y1), radius=5, fill=fill, outline=outline, width=2)
        mask_draw.rounded_rectangle((x0, y0, x1, y1), radius=5, fill=1)
        draw.ellipse((x0 + 2, y0 - 1, x1 - 2, y0 + 4), fill=tuple(min(255, c + 30) for c in fill), outline=outline)
        mask_draw.ellipse((x0 + 2, y0 - 1, x1 - 2, y0 + 4), fill=1)
        label_box = (x0 + 2, y0 + 12, x1 - 2, y0 + 23)

    elif family == "snack":
        x0, x1, y0, y1 = 9 + jitter_x, 39 + jitter_x, 12 + jitter_y, 38 + jitter_y
        points = [(x0 + 4, y0), (x1 - 4, y0), (x1, y0 + 5), (x1 - 2, y1), (x0 + 2, y1), (x0, y0 + 5)]
        draw.polygon(points, fill=fill, outline=outline)
        mask_draw.polygon(points, fill=1)
        draw.line((x0 + 5, y0 + 3, x1 - 5, y0 + 3), fill=outline, width=2)
        draw.line((x0 + 5, y1 - 3, x1 - 5, y1 - 3), fill=outline, width=2)
        label_box = (x0 + 8, y0 + 8, x1 - 8, y0 + 19)

    else:
        x0, x1, y0, y1 = 13 + jitter_x, 35 + jitter_x, 9 + jitter_y, 40 + jitter_y
        body_box = (x0, y0 + 9, x1, y1)
        neck_box = (x0 + 7, y0 + 2, x1 - 7, y0 + 12)
        draw.rounded_rectangle(body_box, radius=5, fill=fill, outline=outline, width=2)
        draw.rectangle(neck_box, fill=fill, outline=outline, width=2)
        mask_draw.rounded_rectangle(body_box, radius=5, fill=1)
        mask_draw.rectangle(neck_box, fill=1)
        draw.rectangle((x0 + 8, y0, x1 - 8, y0 + 4), fill=outline)
        mask_draw.rectangle((x0 + 8, y0, x1 - 8, y0 + 4), fill=1)
        label_box = (x0 + 4, y0 + 17, x1 - 4, y0 + 28)

    return (x0, y0, x1, y1), label_box


def generate_product_image(
    seed,
    family,
    style_id,
    defect,
    camera_num,
    lighting,
    crop_border=True,
    return_masks=False,
):
    """Create a deterministic RGB product image and optional masks."""
    rng = np.random.default_rng(int(seed))
    size = IMG_SIZE
    border = 4

    background = tuple(int(x) for x in rng.integers(226, 241, size=3))
    image = Image.new("RGB", (size, size), background)
    draw = ImageDraw.Draw(image)

    camera_color = CAMERA_COLORS[int(camera_num) % len(CAMERA_COLORS)]
    draw.rectangle((0, 0, size - 1, size - 1), outline=camera_color, width=border)
    pale_camera_color = tuple(min(255, c + 55) for c in camera_color)

    if camera_num % 3 == 0:
        for y in range(8, size, 11):
            draw.line((border, y, size - border, y), fill=pale_camera_color, width=1)
    elif camera_num % 3 == 1:
        for x in range(8, size, 11):
            draw.line((x, border, x, size - border), fill=pale_camera_color, width=1)
    else:
        for x in range(8, size, 12):
            draw.line((x, border, x - 5, size - border), fill=pale_camera_color, width=1)

    product_mask_image = Image.new("L", (size, size), 0)
    defect_mask_image = Image.new("L", (size, size), 0)
    product_mask_draw = ImageDraw.Draw(product_mask_image)
    defect_mask_draw = ImageDraw.Draw(defect_mask_image)

    fill = BODY_COLORS[int(style_id) % len(BODY_COLORS)]
    outline = tuple(max(0, c - 75) for c in fill)
    jitter_x = int(rng.integers(-2, 3))
    jitter_y = int(rng.integers(-2, 3))

    product_box, label_box = _draw_product_shape(
        draw, product_mask_draw, family, fill, outline, jitter_x, jitter_y
    )

    lx0, ly0, lx1, ly1 = label_box
    draw.rounded_rectangle(label_box, radius=2, fill=(246, 246, 235), outline=outline, width=1)
    if family == "beverage":
        draw.ellipse((lx0 + 3, ly0 + 2, lx1 - 3, ly1 - 2), fill=outline)
    elif family == "snack":
        draw.polygon([(lx0 + 2, ly1 - 2), ((lx0 + lx1) // 2, ly0 + 2), (lx1 - 2, ly1 - 2)], fill=outline)
    else:
        draw.rectangle((lx0 + 2, ly0 + 2, lx1 - 2, ly0 + 4), fill=outline)
        draw.rectangle((lx0 + 2, ly0 + 7, lx1 - 5, ly0 + 9), fill=outline)

    defect_type = "none"
    if int(defect) == 1:
        defect_type = ["scratch", "missing_label", "crush"][int(rng.integers(0, 3))]

        if defect_type == "scratch":
            sx0, sy0 = product_box[0] + 2, product_box[1] + 9
            sx1, sy1 = product_box[2] - 2, product_box[3] - 4
            draw.line((sx0, sy0, sx1, sy1), fill=(15, 15, 15), width=4)
            defect_mask_draw.line((sx0, sy0, sx1, sy1), fill=1, width=5)
            draw.line((sx0 + 4, sy0, sx1, sy1 - 6), fill=(235, 235, 235), width=1)
            defect_mask_draw.line((sx0 + 4, sy0, sx1, sy1 - 6), fill=1, width=2)

        elif defect_type == "missing_label":
            draw.rounded_rectangle(label_box, radius=2, fill=fill, outline=(25, 25, 25), width=3)
            defect_mask_draw.rounded_rectangle(label_box, radius=2, fill=1)
            draw.line((lx0 + 2, ly0 + 2, lx1 - 2, ly1 - 2), fill=(25, 25, 25), width=2)
            defect_mask_draw.line((lx0 + 2, ly0 + 2, lx1 - 2, ly1 - 2), fill=1, width=3)

        else:
            center_x = product_box[2] - 1
            center_y = int((product_box[1] + product_box[3]) / 2)
            radius = 6
            draw.ellipse(
                (center_x - radius, center_y - radius, center_x + radius, center_y + radius),
                fill=(25, 25, 25),
                outline=(240, 240, 240),
            )
            defect_mask_draw.ellipse(
                (center_x - radius, center_y - radius, center_x + radius, center_y + radius),
                fill=1,
            )

    if lighting == "dim":
        image = ImageEnhance.Brightness(image).enhance(0.67)
    elif lighting == "warm":
        arr = np.asarray(image).astype(float)
        arr[..., 0] *= 1.12
        arr[..., 2] *= 0.87
        image = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    elif lighting == "glare":
        overlay = Image.new("RGBA", (size, size), (0, 0, 0, 0))
        overlay_draw = ImageDraw.Draw(overlay)
        gx = int(rng.integers(17, 34))
        gy = int(rng.integers(13, 30))
        overlay_draw.ellipse((gx - 9, gy - 6, gx + 10, gy + 7), fill=(255, 255, 255, 110))
        image = Image.alpha_composite(image.convert("RGBA"), overlay).convert("RGB")

    if rng.random() < 0.08:
        image = image.filter(ImageFilter.GaussianBlur(0.45))

    arr = np.asarray(image).astype(float)
    arr += rng.normal(0, 1.8, size=arr.shape)
    image = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

    if crop_border:
        crop_box = (border, border, size - border, size - border)
        image = image.crop(crop_box).resize((size, size), Image.Resampling.BILINEAR)
        product_mask_image = product_mask_image.crop(crop_box).resize((size, size), Image.Resampling.NEAREST)
        defect_mask_image = defect_mask_image.crop(crop_box).resize((size, size), Image.Resampling.NEAREST)

    if return_masks:
        return (
            image,
            np.asarray(product_mask_image, dtype=np.uint8),
            np.asarray(defect_mask_image, dtype=np.uint8),
            defect_type,
        )
    return image


def make_image_metadata(n, source_task=False, camera_ids=range(6), challenge=False, seed=SEED):
    rng = np.random.default_rng(seed)
    defect_rates = {0: 0.08, 1: 0.84, 2: 0.12, 3: 0.80, 4: 0.50, 5: 0.50, 6: 0.50}
    rows = []

    for i in range(n):
        camera_num = int(rng.choice(list(camera_ids)))
        product_family = str(rng.choice(FAMILIES))
        style_id = int(rng.integers(0, len(BODY_COLORS)))
        defect_probability = 0.30 if source_task else defect_rates[camera_num]
        defect = int(rng.random() < defect_probability)

        if challenge:
            lighting = str(rng.choice(["dim", "glare", "warm"], p=[0.45, 0.35, 0.20]))
        else:
            lighting = str(rng.choice(LIGHTINGS, p=[0.67, 0.12, 0.12, 0.09]))

        rows.append({
            "image_id": f"{'source' if source_task else 'target'}_{i:05d}",
            "seed": int(seed * 100000 + i * 17 + camera_num),
            "camera_num": camera_num,
            "camera_id": f"cam_{camera_num}",
            "product_family": product_family,
            "style_id": style_id,
            "lighting": lighting,
            "defect": defect,
            "capture_month": int(rng.integers(1, 7)),
        })

    return pd.DataFrame(rows)


print("Synthetic image generator ready.")

In [ ]:
# ============================================================
# 2.2 Generate source, target, and challenge metadata
# ============================================================
source_meta = make_image_metadata(
    N_SOURCE,
    source_task=True,
    camera_ids=range(6),
    challenge=False,
    seed=SEED + 1,
)

target_meta = make_image_metadata(
    N_TARGET,
    source_task=False,
    camera_ids=range(6),
    challenge=False,
    seed=SEED + 2,
)

challenge_meta = make_image_metadata(
    N_CHALLENGE,
    source_task=False,
    camera_ids=[6],
    challenge=True,
    seed=SEED + 3,
)
challenge_meta["image_id"] = challenge_meta["image_id"].str.replace("target_", "challenge_", regex=False)
challenge_meta["split"] = "challenge"

# The deployment boundary is camera/site. Cameras 0-3 train the model,
# camera 4 selects settings, and camera 5 provides the held-out test.
target_meta["split"] = np.select(
    [
        target_meta["camera_num"].isin([0, 1, 2, 3]),
        target_meta["camera_num"].eq(4),
        target_meta["camera_num"].eq(5),
    ],
    ["train", "validation", "test"],
    default="unused",
)

summary_df = (
    target_meta.groupby(["split", "camera_id"], observed=True)
    .agg(images=("image_id", "size"), defect_rate=("defect", "mean"))
    .reset_index()
)

display(summary_df)
print("Source images:", len(source_meta))
print("Target images:", len(target_meta))
print("Challenge images:", len(challenge_meta))

In [ ]:
# ============================================================
# 2.3 Inspect raw frames, cropped model inputs, and labels
# ============================================================
sample_rows = target_meta.sample(6, random_state=SEED).reset_index(drop=True)

fig, axes = plt.subplots(2, 6, figsize=(13, 5.2))
for j, row in sample_rows.iterrows():
    raw_image = generate_product_image(
        row.seed, row.product_family, row.style_id, row.defect,
        row.camera_num, row.lighting, crop_border=False
    )
    cropped_image = generate_product_image(
        row.seed, row.product_family, row.style_id, row.defect,
        row.camera_num, row.lighting, crop_border=True
    )

    axes[0, j].imshow(raw_image)
    axes[0, j].set_title(f"{row.camera_id}\n{row.product_family}, d={row.defect}")
    axes[0, j].axis("off")

    axes[1, j].imshow(cropped_image)
    axes[1, j].set_title(f"cropped\n{row.lighting}")
    axes[1, j].axis("off")

plt.suptitle("Raw operating context (top) and context-controlled model input (bottom)")
plt.tight_layout()
plt.show()

display(sample_rows[["image_id", "camera_id", "product_family", "lighting", "defect", "split"]])

## 3. Vision task families and output contracts

A business question should determine the model output. Classification is sufficient when one label per image supports the decision. Detection adds locations and counts. Segmentation adds pixel-level area and shape. Document vision returns text and structured fields. Embeddings return reusable vectors for ranking and similarity.

Moving to a more granular output increases annotation cost and implementation complexity. The practical rule is to choose the simplest output that supports the intended action.

In [ ]:
# ============================================================
# 3.1 Map business questions to outputs, metrics, and actions
# ============================================================
vision_task_map = pd.DataFrame([
    {
        "business_question": "Is this package defective?",
        "task_family": "classification",
        "model_output": "one probability per image",
        "primary_metric": "precision, recall, PR-AUC, calibration",
        "downstream_action": "pass, review, or route for rework",
    },
    {
        "business_question": "Where are products and how many are present?",
        "task_family": "object detection",
        "model_output": "class, confidence, and bounding box per object",
        "primary_metric": "precision/recall at an IoU threshold, mAP",
        "downstream_action": "count, locate, and create a store task",
    },
    {
        "business_question": "What share of the shelf is occupied?",
        "task_family": "segmentation",
        "model_output": "pixel mask",
        "primary_metric": "IoU, mean IoU, Dice",
        "downstream_action": "measure area, shape, and compliance",
    },
    {
        "business_question": "What is the invoice total?",
        "task_family": "document vision",
        "model_output": "tokens, coordinates, and structured fields",
        "primary_metric": "field-level exact match",
        "downstream_action": "post, reconcile, or route an exception",
    },
    {
        "business_question": "Which catalog items look similar?",
        "task_family": "embedding retrieval",
        "model_output": "vector per image or region",
        "primary_metric": "Recall@k, Precision@k, NDCG@k",
        "downstream_action": "search, recommend, cluster, or deduplicate",
    },
])

display(vision_task_map)

## 4. Leakage-safe splits and camera-shortcut diagnostics

A random image split is risky when the same store, camera, user, product, session, or near-duplicate scene can appear on both sides of the split. The model may learn context that will not generalize. Here, the target split holds out entire cameras. The challenge set then adds a new camera and difficult lighting.

In [ ]:
# ============================================================
# 4.1 Audit the group-based split
# ============================================================
train_meta = target_meta[target_meta["split"] == "train"].reset_index(drop=True)
val_meta = target_meta[target_meta["split"] == "validation"].reset_index(drop=True)
test_meta = target_meta[target_meta["split"] == "test"].reset_index(drop=True)

# Simulate a realistic labeling budget. Only a stratified subset of the eligible
# training images receives target-task defect labels. The remaining images can
# still support source pretraining, future active learning, or monitoring.
labeled_train_idx, _ = train_test_split(
    np.arange(len(train_meta)),
    train_size=LABELED_TRAIN_FRACTION,
    stratify=train_meta["defect"],
    random_state=SEED,
)
labeled_train_meta = train_meta.iloc[np.sort(labeled_train_idx)].reset_index(drop=True)

split_audit = pd.DataFrame([
    {
        "split": "train",
        "images": len(train_meta),
        "cameras": sorted(train_meta.camera_id.unique().tolist()),
        "defect_rate": train_meta.defect.mean(),
    },
    {
        "split": "validation",
        "images": len(val_meta),
        "cameras": sorted(val_meta.camera_id.unique().tolist()),
        "defect_rate": val_meta.defect.mean(),
    },
    {
        "split": "test",
        "images": len(test_meta),
        "cameras": sorted(test_meta.camera_id.unique().tolist()),
        "defect_rate": test_meta.defect.mean(),
    },
    {
        "split": "challenge",
        "images": len(challenge_meta),
        "cameras": sorted(challenge_meta.camera_id.unique().tolist()),
        "defect_rate": challenge_meta.defect.mean(),
    },
])

display(split_audit)

label_budget_df = pd.DataFrame([{
    "eligible_training_images": len(train_meta),
    "labeled_training_images": len(labeled_train_meta),
    "labeled_fraction": len(labeled_train_meta) / len(train_meta),
    "labeled_defect_rate": labeled_train_meta.defect.mean(),
}])
display(label_budget_df)

assert set(train_meta.camera_id).isdisjoint(set(val_meta.camera_id))
assert set(train_meta.camera_id).isdisjoint(set(test_meta.camera_id))
assert set(val_meta.camera_id).isdisjoint(set(test_meta.camera_id))
print("Camera overlap audit passed.")

In [ ]:
# ============================================================
# 4.2 Camera-only shortcut baseline
# A random split lets a model exploit camera-specific defect rates.
# A held-out-camera test removes that shortcut.
# ============================================================
shortcut_X = target_meta[["camera_id"]]
shortcut_y = target_meta["defect"].to_numpy()

random_train_idx, random_test_idx = train_test_split(
    np.arange(len(target_meta)),
    test_size=0.25,
    stratify=shortcut_y,
    random_state=SEED,
)

def make_camera_shortcut_model():
    preprocessor = ColumnTransformer([
        ("camera", OneHotEncoder(handle_unknown="ignore"), ["camera_id"]),
    ])
    return make_pipeline(preprocessor, LogisticRegression(max_iter=1000, random_state=SEED))

random_shortcut_model = make_camera_shortcut_model()
random_shortcut_model.fit(shortcut_X.iloc[random_train_idx], shortcut_y[random_train_idx])
random_shortcut_prob = random_shortcut_model.predict_proba(shortcut_X.iloc[random_test_idx])[:, 1]

safe_train_idx = target_meta.index[target_meta["split"] == "train"].to_numpy()
safe_test_idx = target_meta.index[target_meta["split"] == "test"].to_numpy()

group_shortcut_model = make_camera_shortcut_model()
group_shortcut_model.fit(shortcut_X.iloc[safe_train_idx], shortcut_y[safe_train_idx])
group_shortcut_prob = group_shortcut_model.predict_proba(shortcut_X.iloc[safe_test_idx])[:, 1]

shortcut_results = pd.DataFrame([
    {
        "evaluation_design": "random image split",
        "information_available": "camera identity only",
        "roc_auc": safe_auc(shortcut_y[random_test_idx], random_shortcut_prob),
        "average_precision": safe_average_precision(shortcut_y[random_test_idx], random_shortcut_prob),
    },
    {
        "evaluation_design": "held-out-camera split",
        "information_available": "camera identity only",
        "roc_auc": safe_auc(shortcut_y[safe_test_idx], group_shortcut_prob),
        "average_precision": safe_average_precision(shortcut_y[safe_test_idx], group_shortcut_prob),
    },
])

display(shortcut_results)

plt.figure(figsize=(7, 4))
plt.bar(shortcut_results["evaluation_design"], shortcut_results["roc_auc"])
plt.ylim(0, 1.05)
plt.ylabel("ROC-AUC")
plt.title("A camera shortcut looks useful under a random split")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 5. CNN classification and staged transfer learning

The target decision is binary: route a visibly defective package for review or rework. We first pretrain a small backbone on a related source task, product-family recognition. We then compare three target workflows: training from scratch, training only a new head on a frozen backbone, and selectively fine-tuning the transferred representation.

This is a classroom-scale substitute for downloading a large pretrained model. Only 35 percent of the eligible target-training images are labeled for the defect task, which makes the label-efficiency question visible. The mechanics are the same: reuse a backbone, replace the head, establish a frozen-feature baseline, and move pretrained weights cautiously with a lower learning rate.

In [ ]:
# ============================================================
# 5.1 Cached image dataset and realistic augmentations
# ============================================================
FAMILY_TO_ID = {name: i for i, name in enumerate(FAMILIES)}


def augment_array(image_uint8):
    """Apply label-preserving augmentations that are plausible for this workflow."""
    arr = image_uint8.astype(np.float32) / 255.0

    if np.random.rand() < 0.50:
        arr = arr[:, ::-1].copy()

    gain = np.random.uniform(0.88, 1.12)
    bias = np.random.uniform(-0.035, 0.035)
    arr = np.clip(arr * gain + bias, 0, 1)

    if np.random.rand() < 0.12:
        height = np.random.randint(3, 7)
        width = np.random.randint(3, 7)
        y0 = np.random.randint(0, IMG_SIZE - height)
        x0 = np.random.randint(0, IMG_SIZE - width)
        arr[y0:y0 + height, x0:x0 + width] = arr.mean(axis=(0, 1))

    return arr


class CachedVisionDataset(Dataset):
    def __init__(self, metadata, target="defect", augment=False, crop_border=True):
        self.df = metadata.reset_index(drop=True).copy()
        self.target = target
        self.augment = augment
        self.crop_border = crop_border

        self.images = np.stack([
            np.asarray(
                generate_product_image(
                    row.seed,
                    row.product_family,
                    row.style_id,
                    row.defect,
                    row.camera_num,
                    row.lighting,
                    crop_border=crop_border,
                ),
                dtype=np.uint8,
            )
            for _, row in self.df.iterrows()
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_uint8 = self.images[idx]
        if self.augment:
            arr = augment_array(image_uint8)
        else:
            arr = image_uint8.astype(np.float32) / 255.0

        x = torch.from_numpy(arr.transpose(2, 0, 1).copy()).float()
        row = self.df.iloc[idx]

        if self.target == "family":
            y = torch.tensor(FAMILY_TO_ID[row.product_family], dtype=torch.long)
        else:
            y = torch.tensor(float(row.defect), dtype=torch.float32)

        return x, y, idx


# Source task split. The target task already has camera-based train/validation/test sets.
source_rng = np.random.default_rng(SEED)
source_order = source_rng.permutation(len(source_meta))
source_cut = int(0.80 * len(source_meta))
source_train_meta = source_meta.iloc[source_order[:source_cut]].reset_index(drop=True)
source_val_meta = source_meta.iloc[source_order[source_cut:]].reset_index(drop=True)

print("Materializing synthetic image tensors...")
source_train_ds = CachedVisionDataset(source_train_meta, target="family", augment=True, crop_border=True)
source_val_ds = CachedVisionDataset(source_val_meta, target="family", augment=False, crop_border=True)
train_ds = CachedVisionDataset(labeled_train_meta, target="defect", augment=True, crop_border=True)
val_ds = CachedVisionDataset(val_meta, target="defect", augment=False, crop_border=True)
test_ds = CachedVisionDataset(test_meta, target="defect", augment=False, crop_border=True)
challenge_ds = CachedVisionDataset(challenge_meta, target="defect", augment=False, crop_border=True)

source_train_loader = DataLoader(source_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
source_val_loader = DataLoader(source_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
challenge_loader = DataLoader(challenge_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Show one original and two independently augmented views.
example_image = train_ds.images[0]
view_a = augment_array(example_image)
view_b = augment_array(example_image)

fig, axes = plt.subplots(1, 3, figsize=(8.5, 3))
for ax, image, title in zip(
    axes,
    [example_image, view_a, view_b],
    ["base image", "augmentation A", "augmentation B"],
):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5.2 Tiny CNN backbone, classifiers, prediction, and training
# ============================================================
class TinyVisionBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.GroupNorm(4, 16),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.GroupNorm(8, 32),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.GroupNorm(8, 64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((3, 3)),
        )
        self.output_dim = 64 * 3 * 3

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return x.flatten(1)


class VisionClassifier(nn.Module):
    def __init__(self, backbone, n_outputs=1):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.output_dim, n_outputs)

    def forward(self, x):
        return self.head(self.backbone(x))


def predict_binary(model, loader):
    model.eval()
    logits_list, labels_list, indices_list = [], [], []
    with torch.no_grad():
        for x, y, idx in loader:
            logits = model(x.to(DEVICE)).view(-1)
            logits_list.append(logits.cpu())
            labels_list.append(y.cpu())
            indices_list.append(idx.cpu())

    logits = torch.cat(logits_list).numpy()
    labels = torch.cat(labels_list).numpy().astype(int)
    indices = torch.cat(indices_list).numpy()
    return logits, sigmoid_np(logits), labels, indices


def predict_multiclass(model, loader):
    model.eval()
    logits_list, labels_list = [], []
    with torch.no_grad():
        for x, y, _ in loader:
            logits_list.append(model(x.to(DEVICE)).cpu())
            labels_list.append(y.cpu())
    logits = torch.cat(logits_list)
    labels = torch.cat(labels_list).numpy()
    probabilities = torch.softmax(logits, dim=1).numpy()
    return probabilities, labels


def train_multiclass(model, train_loader, validation_loader, epochs, lr=2e-3):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    best_score = -np.inf
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for x, y, _ in train_loader:
            optimizer.zero_grad()
            logits = model(x.to(DEVICE))
            loss = loss_fn(logits, y.to(DEVICE))
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(y)

        val_prob, val_y = predict_multiclass(model, validation_loader)
        val_accuracy = accuracy_score(val_y, val_prob.argmax(axis=1))
        row = {
            "epoch": epoch,
            "train_loss": running_loss / len(train_loader.dataset),
            "val_accuracy": val_accuracy,
        }
        history.append(row)
        print(f"source epoch {epoch:02d} | loss={row['train_loss']:.4f} | val_accuracy={val_accuracy:.4f}")

        if val_accuracy > best_score:
            best_score = val_accuracy
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


def train_binary(model, train_loader, validation_loader, epochs, lr=2e-3, parameter_groups=None, name="model"):
    model = model.to(DEVICE)
    trainable = parameter_groups if parameter_groups is not None else [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=1e-4)

    train_labels = train_loader.dataset.df["defect"].to_numpy()
    positives = max(1, train_labels.sum())
    negatives = len(train_labels) - train_labels.sum()
    pos_weight = torch.tensor([negatives / positives], dtype=torch.float32, device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    history = []
    best_score = -np.inf
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for x, y, _ in train_loader:
            optimizer.zero_grad()
            logits = model(x.to(DEVICE)).view(-1)
            loss = loss_fn(logits, y.to(DEVICE))
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(y)

        _, val_prob, val_y, _ = predict_binary(model, validation_loader)
        val_auc = safe_auc(val_y, val_prob)
        val_ap = safe_average_precision(val_y, val_prob)
        row = {
            "epoch": epoch,
            "train_loss": running_loss / len(train_loader.dataset),
            "val_roc_auc": val_auc,
            "val_average_precision": val_ap,
        }
        history.append(row)
        print(
            f"{name} epoch {epoch:02d} | loss={row['train_loss']:.4f} "
            f"| val_auc={val_auc:.4f} | val_ap={val_ap:.4f}"
        )

        if val_auc > best_score:
            best_score = val_auc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


def reset_training_seed(offset=0):
    seed = SEED + int(offset)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print("CNN and training helpers ready.")

In [ ]:
# ============================================================
# 5.3 Pretrain the backbone on product-family recognition
# ============================================================
reset_training_seed(10)
pretrained_model = VisionClassifier(TinyVisionBackbone(), n_outputs=len(FAMILIES))
pretrained_model, pretrain_history = train_multiclass(
    pretrained_model,
    source_train_loader,
    source_val_loader,
    epochs=PRETRAIN_EPOCHS,
    lr=2e-3,
)

plot_training_history(pretrain_history, "val_accuracy", "Source-task pretraining")

source_val_prob, source_val_y = predict_multiclass(pretrained_model, source_val_loader)
source_val_accuracy = accuracy_score(source_val_y, source_val_prob.argmax(axis=1))
print("Restored source-task validation accuracy:", round(source_val_accuracy, 4))

In [ ]:
# ============================================================
# 5.4 Target model A: train a CNN from scratch
# ============================================================
reset_training_seed(20)
scratch_model = VisionClassifier(TinyVisionBackbone(), n_outputs=1)
scratch_model, scratch_history = train_binary(
    scratch_model,
    train_loader,
    val_loader,
    epochs=SCRATCH_EPOCHS,
    lr=2e-3,
    name="scratch",
)

plot_training_history(scratch_history, "val_roc_auc", "Target CNN from scratch")

In [ ]:
# ============================================================
# 5.5 Target model B: freeze the transferred backbone
# ============================================================
reset_training_seed(30)
frozen_model = VisionClassifier(copy.deepcopy(pretrained_model.backbone), n_outputs=1)
for parameter in frozen_model.backbone.parameters():
    parameter.requires_grad = False

frozen_model, frozen_history = train_binary(
    frozen_model,
    train_loader,
    val_loader,
    epochs=FROZEN_EPOCHS,
    lr=3e-3,
    name="frozen_transfer",
)

plot_training_history(frozen_history, "val_roc_auc", "Frozen transferred backbone")

In [ ]:
# ============================================================
# 5.6 Target model C: fine-tune the transferred representation
# The head moves faster, while the backbone uses a smaller rate.
# ============================================================
reset_training_seed(40)
fine_tuned_model = copy.deepcopy(frozen_model)
for parameter in fine_tuned_model.backbone.parameters():
    parameter.requires_grad = True

parameter_groups = [
    {"params": fine_tuned_model.head.parameters(), "lr": 2e-3},
    {"params": fine_tuned_model.backbone.parameters(), "lr": 5e-4},
]

fine_tuned_model, finetune_history = train_binary(
    fine_tuned_model,
    train_loader,
    val_loader,
    epochs=FINETUNE_EPOCHS,
    lr=5e-4,
    parameter_groups=parameter_groups,
    name="fine_tuned_transfer",
)

plot_training_history(finetune_history, "val_roc_auc", "Fine-tuned transferred backbone")

In [ ]:
# ============================================================
# 5.7 Compare target models on the held-out camera
# Thresholds are selected on validation data, not test data.
# ============================================================
models = {
    "cnn_from_scratch": scratch_model,
    "frozen_transfer": frozen_model,
    "fine_tuned_transfer": fine_tuned_model,
}

model_results = []
model_probabilities = {}

for model_name, model in models.items():
    val_logits, val_prob, val_y, _ = predict_binary(model, val_loader)
    test_logits, test_prob, test_y, _ = predict_binary(model, test_loader)
    selected_threshold, selected_val_f1 = find_best_f1_threshold(val_y, val_prob)
    test_metrics = binary_metrics(test_y, test_prob, threshold=selected_threshold)

    model_results.append({
        "model": model_name,
        "validation_selected_f1": selected_val_f1,
        **test_metrics,
    })
    model_probabilities[model_name] = {
        "val_logits": val_logits,
        "val_prob": val_prob,
        "val_y": val_y,
        "test_logits": test_logits,
        "test_prob": test_prob,
        "test_y": test_y,
    }

model_results_df = pd.DataFrame(model_results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
display(model_results_df)

plt.figure(figsize=(8, 4))
plt.bar(model_results_df["model"], model_results_df["roc_auc"])
plt.ylim(0, 1.05)
plt.ylabel("Held-out-camera ROC-AUC")
plt.title("Staged transfer learning comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 6. Calibration, error costs, human review, and slice evaluation

A vision classifier should be treated as a probabilistic measurement system. Ranking metrics tell us whether higher-risk images tend to receive higher scores. Calibration tells us whether the scores have a usable probabilistic meaning. Thresholds then convert those scores into actions, and the correct threshold depends on false-negative cost, false-positive cost, and available review capacity.

In [ ]:
# ============================================================
# 6.1 Calibrate the fine-tuned model with validation logits
# Platt scaling fits a small logistic model on held-out logits.
# ============================================================
selected = model_probabilities["fine_tuned_transfer"]
val_logits = selected["val_logits"]
val_prob_raw = selected["val_prob"]
val_y = selected["val_y"]
test_logits = selected["test_logits"]
test_prob_raw = selected["test_prob"]
test_y = selected["test_y"]

platt_model = LogisticRegression(C=1000.0, max_iter=2000, random_state=SEED)
platt_model.fit(val_logits.reshape(-1, 1), val_y)

val_prob_calibrated = platt_model.predict_proba(val_logits.reshape(-1, 1))[:, 1]
test_prob_calibrated = platt_model.predict_proba(test_logits.reshape(-1, 1))[:, 1]

calibration_summary = pd.DataFrame([
    {
        "probability_version": "raw",
        "test_brier": brier_score_loss(test_y, test_prob_raw),
        "test_ece": expected_calibration_error(test_y, test_prob_raw),
        "test_roc_auc": safe_auc(test_y, test_prob_raw),
    },
    {
        "probability_version": "Platt calibrated",
        "test_brier": brier_score_loss(test_y, test_prob_calibrated),
        "test_ece": expected_calibration_error(test_y, test_prob_calibrated),
        "test_roc_auc": safe_auc(test_y, test_prob_calibrated),
    },
])

display(calibration_summary)
plot_reliability(
    test_y,
    {"raw": test_prob_raw, "Platt calibrated": test_prob_calibrated},
    title="Held-out-camera reliability",
)

In [ ]:
# ============================================================
# 6.2 Choose a threshold using business error costs
# ============================================================
FALSE_NEGATIVE_COST = 8.0
FALSE_POSITIVE_COST = 1.0

cost_threshold, validation_cost_curve = find_min_cost_threshold(
    val_y,
    val_prob_calibrated,
    false_negative_cost=FALSE_NEGATIVE_COST,
    false_positive_cost=FALSE_POSITIVE_COST,
)

cost_based_test_metrics = binary_metrics(test_y, test_prob_calibrated, threshold=cost_threshold)
display(pd.DataFrame([{
    "false_negative_cost": FALSE_NEGATIVE_COST,
    "false_positive_cost": FALSE_POSITIVE_COST,
    **cost_based_test_metrics,
}]))

plt.figure(figsize=(7, 4))
plt.plot(validation_cost_curve["threshold"], validation_cost_curve["total_cost"])
plt.axvline(cost_threshold, linestyle="--", label=f"selected={cost_threshold:.2f}")
plt.xlabel("Decision threshold")
plt.ylabel("Validation error cost")
plt.title("Threshold selection is a business decision")
plt.legend()
plt.show()

plot_confusion_matrix(
    test_y,
    test_prob_calibrated,
    threshold=cost_threshold,
    title="Cost-based policy on held-out camera",
)

In [ ]:
# ============================================================
# 6.3 Two-threshold triage under a review budget
# The most uncertain validation cases define a review band.
# ============================================================
TARGET_REVIEW_RATE = 0.25
validation_uncertainty = np.abs(val_prob_calibrated - 0.50)
review_radius = float(np.quantile(validation_uncertainty, TARGET_REVIEW_RATE))
LOW_THRESHOLD = max(0.0, 0.50 - review_radius)
HIGH_THRESHOLD = min(1.0, 0.50 + review_radius)


def triage_summary(y_true, prob, low_threshold, high_threshold):
    route = np.where(
        prob >= high_threshold,
        "auto_route_defect",
        np.where(prob <= low_threshold, "auto_pass", "human_review"),
    )
    summary = {
        "low_threshold": low_threshold,
        "high_threshold": high_threshold,
        "review_rate": float((route == "human_review").mean()),
        "auto_route_rate": float((route == "auto_route_defect").mean()),
        "auto_pass_rate": float((route == "auto_pass").mean()),
    }

    auto_positive = route == "auto_route_defect"
    auto_negative = route == "auto_pass"
    summary["auto_route_precision"] = float(y_true[auto_positive].mean()) if auto_positive.any() else np.nan
    summary["auto_pass_miss_rate"] = float(y_true[auto_negative].mean()) if auto_negative.any() else np.nan
    return summary, route

triage_metrics, test_routes = triage_summary(
    test_y,
    test_prob_calibrated,
    LOW_THRESHOLD,
    HIGH_THRESHOLD,
)

display(pd.DataFrame([triage_metrics]))

In [ ]:
# ============================================================
# 6.4 Slice-aware evaluation and challenge-set performance
# ============================================================
challenge_logits, challenge_prob_raw, challenge_y, _ = predict_binary(fine_tuned_model, challenge_loader)
challenge_prob_calibrated = platt_model.predict_proba(challenge_logits.reshape(-1, 1))[:, 1]

test_predictions = test_meta.copy()
test_predictions["dataset"] = "held_out_camera"
test_predictions["probability"] = test_prob_calibrated
test_predictions["prediction"] = (test_prob_calibrated >= cost_threshold).astype(int)
test_predictions["route"] = test_routes

challenge_predictions = challenge_meta.copy()
challenge_predictions["dataset"] = "challenge"
challenge_predictions["probability"] = challenge_prob_calibrated
challenge_predictions["prediction"] = (challenge_prob_calibrated >= cost_threshold).astype(int)
_, challenge_routes = triage_summary(
    challenge_y,
    challenge_prob_calibrated,
    LOW_THRESHOLD,
    HIGH_THRESHOLD,
)
challenge_predictions["route"] = challenge_routes

combined_predictions = pd.concat([test_predictions, challenge_predictions], ignore_index=True)


def grouped_slice_metrics(df, slice_col):
    rows = []
    for (dataset_name, slice_value), group in df.groupby(["dataset", slice_col], observed=True):
        metrics = binary_metrics(group["defect"].to_numpy(), group["probability"].to_numpy(), threshold=cost_threshold)
        rows.append({
            "dataset": dataset_name,
            "slice": slice_col,
            "slice_value": slice_value,
            "n": len(group),
            **metrics,
        })
    return pd.DataFrame(rows)

slice_metrics_df = pd.concat([
    grouped_slice_metrics(combined_predictions, "lighting"),
    grouped_slice_metrics(combined_predictions, "product_family"),
], ignore_index=True)

challenge_summary = pd.DataFrame([
    {"dataset": "held_out_camera", **binary_metrics(test_y, test_prob_calibrated, cost_threshold)},
    {"dataset": "challenge", **binary_metrics(challenge_y, challenge_prob_calibrated, cost_threshold)},
])

display(challenge_summary)
display(slice_metrics_df.sort_values(["dataset", "slice", "roc_auc"]).reset_index(drop=True))

In [ ]:
# ============================================================
# 6.5 Inspect high-confidence errors as an error-analysis queue
# ============================================================
error_df = combined_predictions[combined_predictions["prediction"] != combined_predictions["defect"]].copy()
error_df["confidence"] = np.where(
    error_df["prediction"] == 1,
    error_df["probability"],
    1 - error_df["probability"],
)
error_df = error_df.sort_values("confidence", ascending=False).head(6)

if len(error_df) == 0:
    print("No errors at the selected threshold in this run.")
else:
    fig, axes = plt.subplots(1, len(error_df), figsize=(2.4 * len(error_df), 2.8))
    if len(error_df) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, error_df.iterrows()):
        image = generate_product_image(
            row.seed,
            row.product_family,
            row.style_id,
            row.defect,
            row.camera_num,
            row.lighting,
            crop_border=True,
        )
        ax.imshow(image)
        ax.set_title(f"y={row.defect}, p={row.probability:.2f}\n{row.dataset}, {row.lighting}")
        ax.axis("off")
    plt.suptitle("High-confidence errors for review")
    plt.tight_layout()
    plt.show()

    display(error_df[[
        "image_id", "dataset", "product_family", "lighting", "defect", "prediction", "probability", "confidence"
    ]])

## 7. Detection and segmentation outputs

Classification answers whether an image contains a problem. Detection and segmentation add location. The next demonstration uses known synthetic ground truth and intentionally imperfect predictions. This isolates the output schema and evaluation logic without turning the chapter into a full detection-framework tutorial.

In [ ]:
# ============================================================
# 7.1 Create a shelf scene with object boxes and a semantic mask
# ============================================================
def make_shelf_scene(seed=SEED, width=180, height=110):
    rng = np.random.default_rng(seed)
    image = Image.new("RGB", (width, height), (235, 235, 230))
    draw = ImageDraw.Draw(image)

    for y in [15, 58, 101]:
        draw.line((5, y, width - 5, y), fill=(110, 110, 110), width=2)

    instance_mask = np.zeros((height, width), dtype=np.int32)
    boxes = []
    object_id = 1

    slots = []
    for row, y0 in enumerate([20, 63]):
        for col, x0 in enumerate([12, 44, 76, 108, 140]):
            slots.append((row, col, x0, y0))

    missing_slot = (1, 2)
    for row, col, x0, y0 in slots:
        if (row, col) == missing_slot:
            continue

        width_i = int(rng.integers(21, 27))
        height_i = int(rng.integers(28, 34))
        x1 = min(width - 6, x0 + width_i)
        y1 = min(height - 8, y0 + height_i)
        fill = BODY_COLORS[object_id % len(BODY_COLORS)]
        outline = tuple(max(0, c - 70) for c in fill)
        draw.rounded_rectangle((x0, y0, x1, y1), radius=3, fill=fill, outline=outline, width=2)
        draw.rectangle((x0 + 4, y0 + 9, x1 - 4, y0 + 18), fill=(245, 245, 235), outline=outline)

        instance_mask[y0:y1 + 1, x0:x1 + 1] = object_id
        boxes.append({
            "object_id": object_id,
            "class_name": "package",
            "x1": x0,
            "y1": y0,
            "x2": x1,
            "y2": y1,
        })
        object_id += 1

    return image, pd.DataFrame(boxes), instance_mask


def perturb_detections(ground_truth_boxes, seed=SEED + 10):
    rng = np.random.default_rng(seed)
    predictions = []
    for _, row in ground_truth_boxes.iterrows():
        if rng.random() < 0.12:
            continue
        jitter = rng.integers(-3, 4, size=4)
        predictions.append({
            "class_name": row.class_name,
            "confidence": float(rng.uniform(0.68, 0.97)),
            "x1": int(row.x1 + jitter[0]),
            "y1": int(row.y1 + jitter[1]),
            "x2": int(row.x2 + jitter[2]),
            "y2": int(row.y2 + jitter[3]),
        })

    predictions.append({
        "class_name": "package",
        "confidence": 0.58,
        "x1": 148,
        "y1": 25,
        "x2": 171,
        "y2": 51,
    })
    return pd.DataFrame(predictions).sort_values("confidence", ascending=False).reset_index(drop=True)


shelf_image, gt_boxes, gt_instance_mask = make_shelf_scene()
pred_boxes = perturb_detections(gt_boxes)

gt_semantic_mask = gt_instance_mask > 0
pred_semantic_mask = np.roll(gt_semantic_mask, shift=2, axis=1)
pred_semantic_mask = binary_dilation(pred_semantic_mask, iterations=1)
pred_semantic_mask[67:92, 75:101] = False

print("Ground-truth objects:", len(gt_boxes))
print("Predicted objects:", len(pred_boxes))

In [ ]:
# ============================================================
# 7.2 Visualize boxes and masks
# ============================================================
fig, ax = plt.subplots(figsize=(9, 5))
ax.imshow(shelf_image)
for _, row in gt_boxes.iterrows():
    ax.add_patch(Rectangle((row.x1, row.y1), row.x2 - row.x1, row.y2 - row.y1, fill=False, linewidth=2))
ax.set_title("Ground-truth detection boxes")
ax.axis("off")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.imshow(shelf_image)
for _, row in pred_boxes.iterrows():
    ax.add_patch(Rectangle((row.x1, row.y1), row.x2 - row.x1, row.y2 - row.y1, fill=False, linewidth=2))
    ax.text(row.x1, max(0, row.y1 - 2), f"{row.confidence:.2f}", fontsize=8)
ax.set_title("Predicted detection boxes")
ax.axis("off")
plt.show()

plt.figure(figsize=(7, 4))
plt.imshow(gt_semantic_mask)
plt.title("Ground-truth segmentation mask")
plt.axis("off")
plt.show()

plt.figure(figsize=(7, 4))
plt.imshow(pred_semantic_mask)
plt.title("Predicted segmentation mask")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# 7.3 IoU, Dice, detection matching, and operational measures
# ============================================================
def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_x1, inter_y1 = max(ax1, bx1), max(ay1, by1)
    inter_x2, inter_y2 = min(ax2, bx2), min(ay2, by2)
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    intersection = inter_w * inter_h
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def evaluate_detection(gt_df, pred_df, iou_threshold=0.50):
    matched_gt = set()
    matched_ious = []
    true_positives = 0

    for pred_idx, pred in pred_df.sort_values("confidence", ascending=False).iterrows():
        best_gt_idx = None
        best_iou = 0.0
        pred_box = (pred.x1, pred.y1, pred.x2, pred.y2)

        for gt_idx, gt in gt_df.iterrows():
            if gt_idx in matched_gt:
                continue
            gt_box = (gt.x1, gt.y1, gt.x2, gt.y2)
            current_iou = box_iou(pred_box, gt_box)
            if current_iou > best_iou:
                best_iou = current_iou
                best_gt_idx = gt_idx

        if best_gt_idx is not None and best_iou >= iou_threshold:
            matched_gt.add(best_gt_idx)
            matched_ious.append(best_iou)
            true_positives += 1

    false_positives = len(pred_df) - true_positives
    false_negatives = len(gt_df) - true_positives
    precision = true_positives / max(1, true_positives + false_positives)
    recall = true_positives / max(1, true_positives + false_negatives)

    return {
        "iou_threshold": iou_threshold,
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "precision": precision,
        "recall": recall,
        "mean_matched_iou": float(np.mean(matched_ious)) if matched_ious else np.nan,
    }


def mask_iou_and_dice(y_true_mask, y_pred_mask):
    y_true_mask = y_true_mask.astype(bool)
    y_pred_mask = y_pred_mask.astype(bool)
    intersection = np.logical_and(y_true_mask, y_pred_mask).sum()
    union = np.logical_or(y_true_mask, y_pred_mask).sum()
    iou = intersection / union if union else 1.0
    dice = 2 * intersection / (y_true_mask.sum() + y_pred_mask.sum()) if (y_true_mask.sum() + y_pred_mask.sum()) else 1.0
    return iou, dice


detection_metrics = evaluate_detection(gt_boxes, pred_boxes, iou_threshold=0.50)
segmentation_iou, segmentation_dice = mask_iou_and_dice(gt_semantic_mask, pred_semantic_mask)

spatial_metrics_df = pd.DataFrame([
    {"task": "detection", **detection_metrics},
    {
        "task": "segmentation",
        "iou_threshold": np.nan,
        "true_positives": np.nan,
        "false_positives": np.nan,
        "false_negatives": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "mean_matched_iou": segmentation_iou,
        "dice": segmentation_dice,
    },
])

display(spatial_metrics_df)

operational_spatial_output = pd.DataFrame([
    {
        "measure": "package count",
        "ground_truth": len(gt_boxes),
        "prediction": len(pred_boxes),
        "absolute_error": abs(len(gt_boxes) - len(pred_boxes)),
        "possible_action": "verify the shelf when count error exceeds tolerance",
    },
    {
        "measure": "occupied shelf area",
        "ground_truth": gt_semantic_mask.mean(),
        "prediction": pred_semantic_mask.mean(),
        "absolute_error": abs(gt_semantic_mask.mean() - pred_semantic_mask.mean()),
        "possible_action": "compare shelf-share estimates with planogram targets",
    },
])

display(operational_spatial_output)

## 8. Document vision and field-level accuracy

OCR returns text, but many business workflows need a field with the correct role, such as the final invoice total rather than the first currency amount on the page. Layout-aware document vision preserves token coordinates so that extraction can use both text and spatial relationships.

The code below assumes that an OCR system has already returned tokens and bounding boxes. It compares a naive text-only rule with an anchor-and-layout rule.

In [ ]:
# ============================================================
# 8.1 Render a synthetic invoice and return OCR-like tokens
# ============================================================
def build_invoice(invoice_id, subtotal, tax, corrupt_total=False):
    total = subtotal + tax
    tokens = [
        {"text": "INVOICE", "x1": 12, "y1": 8, "x2": 58, "y2": 18},
        {"text": invoice_id, "x1": 113, "y1": 8, "x2": 162, "y2": 18},
        {"text": "Service", "x1": 12, "y1": 35, "x2": 58, "y2": 45},
        {"text": f"${subtotal:.2f}", "x1": 116, "y1": 35, "x2": 166, "y2": 45},
        {"text": "Tax", "x1": 12, "y1": 55, "x2": 38, "y2": 65},
        {"text": f"${tax:.2f}", "x1": 116, "y1": 55, "x2": 166, "y2": 65},
        {"text": "TOTAL", "x1": 12, "y1": 80, "x2": 52, "y2": 92},
        {"text": f"${total:.2f}", "x1": 112, "y1": 80, "x2": 166, "y2": 92},
    ]

    if corrupt_total:
        tokens[-1]["text"] = tokens[-1]["text"].replace("0", "O", 1)

    image = Image.new("RGB", (180, 110), (250, 250, 248))
    draw = ImageDraw.Draw(image)
    draw.rectangle((5, 5, 175, 105), outline=(100, 100, 100), width=1)
    draw.line((10, 74, 170, 74), fill=(130, 130, 130), width=1)
    for token in tokens:
        draw.text((token["x1"], token["y1"]), token["text"], fill=(20, 20, 20))

    return image, pd.DataFrame(tokens), f"${total:.2f}"


invoice_image, invoice_tokens, invoice_truth = build_invoice("INV-1042", subtotal=125.0, tax=10.0)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.imshow(invoice_image)
for _, token in invoice_tokens.iterrows():
    ax.add_patch(Rectangle(
        (token.x1, token.y1),
        token.x2 - token.x1,
        token.y2 - token.y1,
        fill=False,
        linewidth=1,
    ))
ax.set_title("Document image with OCR-style token boxes")
ax.axis("off")
plt.show()

display(invoice_tokens)

In [ ]:
# ============================================================
# 8.2 Text-only extraction versus layout-aware extraction
# ============================================================
def looks_like_currency(text):
    return isinstance(text, str) and text.startswith("$") and len(text) >= 4


def naive_first_amount(tokens_df):
    candidates = tokens_df[tokens_df["text"].map(looks_like_currency)]
    return candidates.iloc[0]["text"] if len(candidates) else None


def extract_amount_near_anchor(tokens_df, anchor="TOTAL"):
    anchors = tokens_df[tokens_df["text"].str.upper() == anchor.upper()]
    amounts = tokens_df[tokens_df["text"].map(looks_like_currency)].copy()
    if len(anchors) == 0 or len(amounts) == 0:
        return None

    anchor_row = anchors.iloc[0]
    anchor_center_y = (anchor_row.y1 + anchor_row.y2) / 2
    amounts["center_y"] = (amounts["y1"] + amounts["y2"]) / 2
    amounts["vertical_distance"] = np.abs(amounts["center_y"] - anchor_center_y)
    amounts["is_right_of_anchor"] = amounts["x1"] > anchor_row.x2

    eligible = amounts[amounts["is_right_of_anchor"]]
    if len(eligible) == 0:
        eligible = amounts
    return eligible.sort_values(["vertical_distance", "x1"]).iloc[0]["text"]


single_invoice_comparison = pd.DataFrame([{
    "ground_truth_total": invoice_truth,
    "naive_text_only": naive_first_amount(invoice_tokens),
    "layout_aware": extract_amount_near_anchor(invoice_tokens),
}])
display(single_invoice_comparison)

# Evaluate a small document batch. Two totals contain simulated OCR character errors.
rng = np.random.default_rng(SEED)
document_rows = []
for i in range(12):
    subtotal = float(rng.integers(45, 240))
    tax = round(subtotal * float(rng.uniform(0.05, 0.10)), 2)
    corrupt = i in {4, 10}
    _, tokens, truth = build_invoice(f"INV-{2000 + i}", subtotal, tax, corrupt_total=corrupt)
    naive_value = naive_first_amount(tokens)
    layout_value = extract_amount_near_anchor(tokens)
    document_rows.append({
        "invoice_id": f"INV-{2000 + i}",
        "ground_truth": truth,
        "naive_value": naive_value,
        "layout_value": layout_value,
        "naive_exact_match": int(naive_value == truth),
        "layout_exact_match": int(layout_value == truth),
        "simulated_ocr_error": corrupt,
    })

document_results_df = pd.DataFrame(document_rows)
document_metric_summary = pd.DataFrame([
    {"method": "naive text-only", "field_exact_match": document_results_df["naive_exact_match"].mean()},
    {"method": "layout-aware anchor", "field_exact_match": document_results_df["layout_exact_match"].mean()},
])

display(document_metric_summary)
display(document_results_df)

## 9. Vision embeddings for search and recommendation

A classifier produces a task-specific label, while a backbone produces a reusable representation. We use the pretrained product-family backbone as an encoder, normalize its vectors, and perform cosine-similarity retrieval. Ranking metrics ask whether useful items appear near the top, not whether a single class label is correct.

In [ ]:
# ============================================================
# 9.1 Extract normalized embeddings for a gallery
# ============================================================
gallery_meta = pd.concat([val_meta, test_meta], ignore_index=True)
gallery_ds = CachedVisionDataset(gallery_meta, target="defect", augment=False, crop_border=True)
gallery_loader = DataLoader(gallery_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


def extract_embeddings(backbone, loader):
    backbone.eval()
    vectors, indices = [], []
    with torch.no_grad():
        for x, _, idx in loader:
            embedding = backbone(x.to(DEVICE)).cpu().numpy()
            vectors.append(embedding)
            indices.append(idx.numpy())
    vectors = np.vstack(vectors)
    indices = np.concatenate(indices)
    vectors = vectors / np.clip(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12, None)
    return vectors, indices


visual_embeddings, gallery_indices = extract_embeddings(pretrained_model.backbone, gallery_loader)
print("Embedding matrix shape:", visual_embeddings.shape)
print("Average vector norm:", np.linalg.norm(visual_embeddings, axis=1).mean().round(4))

In [ ]:
# ============================================================
# 9.2 Evaluate family-level retrieval with ranking metrics
# ============================================================
def ndcg_at_k(relevance, k):
    relevance = np.asarray(relevance[:k], dtype=float)
    discounts = 1.0 / np.log2(np.arange(2, len(relevance) + 2))
    dcg = float((relevance * discounts).sum())
    ideal = np.sort(relevance)[::-1]
    idcg = float((ideal * discounts).sum())
    return dcg / idcg if idcg > 0 else 0.0


similarity_matrix = visual_embeddings @ visual_embeddings.T
np.fill_diagonal(similarity_matrix, -np.inf)

retrieval_rows = []
for query_idx in range(len(gallery_meta)):
    ranking = np.argsort(-similarity_matrix[query_idx])
    query_family = gallery_meta.iloc[query_idx]["product_family"]
    relevance = (gallery_meta.iloc[ranking]["product_family"].to_numpy() == query_family).astype(int)
    retrieval_rows.append({
        "query_idx": query_idx,
        "hit_at_1": int(relevance[0] == 1),
        "precision_at_5": float(relevance[:5].mean()),
        "ndcg_at_5": ndcg_at_k(relevance, 5),
    })

retrieval_query_df = pd.DataFrame(retrieval_rows)
retrieval_summary_df = pd.DataFrame([{
    "queries": len(retrieval_query_df),
    "hit_at_1": retrieval_query_df["hit_at_1"].mean(),
    "precision_at_5": retrieval_query_df["precision_at_5"].mean(),
    "ndcg_at_5": retrieval_query_df["ndcg_at_5"].mean(),
}])

display(retrieval_summary_df)

In [ ]:
# ============================================================
# 9.3 Inspect one visual-search result set
# ============================================================
query_idx = int(np.where(gallery_meta["product_family"].to_numpy() == "cleaner")[0][0])
neighbor_order = np.argsort(-similarity_matrix[query_idx])[:5]

fig, axes = plt.subplots(1, 6, figsize=(13, 2.8))
axes[0].imshow(gallery_ds.images[query_idx])
axes[0].set_title(f"query\n{gallery_meta.iloc[query_idx].product_family}")
axes[0].axis("off")

neighbor_rows = []
for rank, (ax, neighbor_idx) in enumerate(zip(axes[1:], neighbor_order), start=1):
    row = gallery_meta.iloc[neighbor_idx]
    similarity = similarity_matrix[query_idx, neighbor_idx]
    ax.imshow(gallery_ds.images[neighbor_idx])
    ax.set_title(f"#{rank} {row.product_family}\ncos={similarity:.3f}")
    ax.axis("off")
    neighbor_rows.append({
        "rank": rank,
        "image_id": row.image_id,
        "product_family": row.product_family,
        "cosine_similarity": similarity,
        "relevant": int(row.product_family == gallery_meta.iloc[query_idx].product_family),
    })

plt.suptitle("Embedding-based visual search")
plt.tight_layout()
plt.show()

display(pd.DataFrame(neighbor_rows))

## 10. Self-supervision, patch tokens, and foundation model strategy

Self-supervised vision learning creates multiple views of the same image and trains an encoder to preserve identity-relevant information across those views. Vision transformers divide an image into patches and treat the patch embeddings as a sequence. Vision-language foundation models add paired text and image training, enabling cross-modal retrieval and zero-shot prompting.

The demonstrations below focus on the representation logic. The optional CLIP cell is separated because it requires a model download and should not be necessary for the core classroom workflow.

In [ ]:
# ============================================================
# 10.1 Contrastive-learning intuition with two views
# ============================================================
base_uint8 = gallery_ds.images[query_idx]
view_1 = base_uint8.astype(np.float32) / 255.0
view_2 = augment_array(base_uint8)
query_family = gallery_meta.iloc[query_idx]["product_family"]
different_candidates = np.where(gallery_meta["product_family"].to_numpy() != query_family)[0]
different_idx = int(different_candidates[0])
different_uint8 = gallery_ds.images[different_idx].astype(np.float32) / 255.0


def embed_arrays(backbone, arrays):
    tensor = torch.from_numpy(np.stack(arrays).transpose(0, 3, 1, 2).copy()).float().to(DEVICE)
    backbone.eval()
    with torch.no_grad():
        vectors = backbone(tensor).cpu().numpy()
    vectors = vectors / np.clip(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12, None)
    return vectors


view_embeddings = embed_arrays(pretrained_model.backbone, [view_1, view_2, different_uint8])
same_image_similarity = float(view_embeddings[0] @ view_embeddings[1])
different_image_similarity = float(view_embeddings[0] @ view_embeddings[2])

fig, axes = plt.subplots(1, 3, figsize=(8.5, 3))
for ax, image, title in zip(
    axes,
    [view_1, view_2, different_uint8],
    ["base view", "augmented view", "different family"],
):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

contrastive_demo_df = pd.DataFrame([
    {"comparison": "two augmented views of the same image", "cosine_similarity": same_image_similarity},
    {"comparison": "augmented view versus a different image", "cosine_similarity": different_image_similarity},
])
display(contrastive_demo_df)

In [ ]:
# ============================================================
# 10.2 Turn an image into Vision Transformer patch tokens
# ============================================================
def patchify(image_array, patch_size=8):
    height, width, channels = image_array.shape
    assert height % patch_size == 0 and width % patch_size == 0
    patches = image_array.reshape(
        height // patch_size,
        patch_size,
        width // patch_size,
        patch_size,
        channels,
    ).transpose(0, 2, 1, 3, 4)
    return patches.reshape(-1, patch_size, patch_size, channels)


patch_size = 8
patches = patchify(base_uint8, patch_size=patch_size)
patch_vectors = patches.reshape(len(patches), -1).astype(np.float32) / 255.0

print("Image shape:", base_uint8.shape)
print("Patch size:", patch_size)
print("Number of patch tokens:", len(patches))
print("Flattened patch-vector shape:", patch_vectors.shape)
print("Standard self-attention pair comparisons:", len(patches) ** 2)

plt.figure(figsize=(3, 3))
plt.imshow(base_uint8)
plt.title("Original image")
plt.axis("off")
plt.show()

fig, axes = plt.subplots(6, 6, figsize=(7, 7))
for ax, patch, patch_id in zip(axes.flat, patches, range(len(patches))):
    ax.imshow(patch)
    ax.set_title(str(patch_id), fontsize=7)
    ax.axis("off")
plt.suptitle("Image patches become a token sequence")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 10.3 Make-versus-buy and representation strategy
# ============================================================
vision_strategy_df = pd.DataFrame([
    {
        "situation": "rapid MVP with common objects or generic OCR",
        "starting_strategy": "managed API",
        "advantage": "fast workflow and ROI validation",
        "main_risk": "per-call cost, privacy, lock-in, limited domain control",
        "advance_when": "domain error or production economics exceed tolerance",
    },
    {
        "situation": "small labeled dataset and stable categories",
        "starting_strategy": "frozen pretrained backbone plus new head",
        "advantage": "low variance and low training cost",
        "main_risk": "underfitting when domain features differ",
        "advance_when": "error analysis shows missing domain-specific features",
    },
    {
        "situation": "moderate labels and domain-specific visual cues",
        "starting_strategy": "partial or full fine-tuning",
        "advantage": "adapts reusable features to the task",
        "main_risk": "overfitting or catastrophic forgetting",
        "advance_when": "unlabeled in-domain images are abundant or labels remain costly",
    },
    {
        "situation": "many unlabeled images and several downstream tasks",
        "starting_strategy": "self-supervised or foundation-model embeddings",
        "advantage": "one reusable representation supports many uses",
        "main_risk": "domain mismatch, bias, prompt sensitivity, governance burden",
        "advance_when": "task-specific validation justifies deployment",
    },
])

display(vision_strategy_df)

In [ ]:
# ============================================================
# 10.4 Optional vision-language foundation model demonstration
# This cell is skipped by default. It downloads public CLIP weights.
# ============================================================
if RUN_OPTIONAL_CLIP_DEMO:
    ensure("transformers")
    from transformers import CLIPModel, CLIPProcessor

    model_id = "openai/clip-vit-base-patch32"
    clip_model = CLIPModel.from_pretrained(model_id).to(DEVICE)
    clip_processor = CLIPProcessor.from_pretrained(model_id)

    prompts = [
        "a photo of a beverage can",
        "a photo of a snack package",
        "a photo of a cleaning bottle",
    ]
    optional_image = Image.fromarray(base_uint8)
    inputs = clip_processor(text=prompts, images=optional_image, return_tensors="pt", padding=True)
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = clip_model(**inputs)
        probabilities = outputs.logits_per_image.softmax(dim=1).cpu().numpy()[0]

    display(pd.DataFrame({"prompt": prompts, "zero_shot_probability": probabilities}).sort_values(
        "zero_shot_probability", ascending=False
    ))
else:
    print("Optional CLIP demonstration skipped. Set RUN_OPTIONAL_CLIP_DEMO = True to download and run it.")

## 11. Monitoring, governance, and saved artifacts

Production reliability requires more than a static test score. Input quality can shift because of lighting, blur, resolution, camera movement, or packaging redesign. Prediction distributions and review workload can drift even before labels arrive. Slice metrics identify localized failures after labels become available.

The monitoring table below compares the held-out camera with the harder challenge batch. The governance record then documents the decision target, split logic, thresholds, privacy controls, and known limitations.

In [ ]:
# ============================================================
# 11.1 Input quality and prediction-health monitoring
# ============================================================
def image_quality_features(images_uint8):
    arr = images_uint8.astype(np.float32) / 255.0
    grayscale = arr.mean(axis=3)
    horizontal_change = np.abs(np.diff(grayscale, axis=2)).mean(axis=(1, 2))
    vertical_change = np.abs(np.diff(grayscale, axis=1)).mean(axis=(1, 2))
    return pd.DataFrame({
        "brightness": arr.mean(axis=(1, 2, 3)),
        "contrast": arr.std(axis=(1, 2, 3)),
        "sharpness_proxy": horizontal_change + vertical_change,
    })


def summarize_monitoring_batch(name, quality_df, probabilities, routes):
    return {
        "batch": name,
        "n_images": len(quality_df),
        "mean_brightness": quality_df["brightness"].mean(),
        "mean_contrast": quality_df["contrast"].mean(),
        "mean_sharpness_proxy": quality_df["sharpness_proxy"].mean(),
        "mean_defect_probability": np.mean(probabilities),
        "high_confidence_rate": np.mean((probabilities <= 0.10) | (probabilities >= 0.90)),
        "review_rate": np.mean(routes == "human_review"),
    }


test_quality = image_quality_features(test_ds.images)
challenge_quality = image_quality_features(challenge_ds.images)

monitoring_summary_df = pd.DataFrame([
    summarize_monitoring_batch("held_out_camera", test_quality, test_prob_calibrated, test_routes),
    summarize_monitoring_batch("challenge", challenge_quality, challenge_prob_calibrated, challenge_routes),
])

display(monitoring_summary_df)

plt.figure(figsize=(7, 4))
plt.hist(test_quality["brightness"], bins=18, alpha=0.60, label="held-out camera")
plt.hist(challenge_quality["brightness"], bins=18, alpha=0.60, label="challenge")
plt.xlabel("Mean image brightness")
plt.ylabel("Image count")
plt.title("Input-quality shift")
plt.legend()
plt.show()

monitoring_actions_df = pd.DataFrame([
    {
        "monitoring_layer": "data quality",
        "signal": "brightness, contrast, sharpness, missing frames, resolution",
        "possible_response": "repair capture process, alert operations, or block invalid inputs",
    },
    {
        "monitoring_layer": "prediction health",
        "signal": "confidence distribution, positive rate, anomaly rate",
        "possible_response": "review samples, recalibrate, or adjust thresholds",
    },
    {
        "monitoring_layer": "slice performance",
        "signal": "metrics by camera, site, lighting, product, and region",
        "possible_response": "targeted data collection or local validation",
    },
    {
        "monitoring_layer": "human review workload",
        "signal": "queue size, override rate, time to resolution",
        "possible_response": "rebalance thresholds, improve reviewer interface, refine labels",
    },
])

display(monitoring_actions_df)

In [ ]:
# ============================================================
# 11.2 Privacy, reliability, and system-card artifacts
# ============================================================
governance_checklist_df = pd.DataFrame([
    {
        "risk_area": "data minimization",
        "required_control": "crop to the needed product region and avoid retaining unrelated people or surroundings",
        "evidence_to_keep": "capture specification and retention policy",
    },
    {
        "risk_area": "consent and usage boundaries",
        "required_control": "document why images were collected and which uses are permitted",
        "evidence_to_keep": "data-use approval and access log",
    },
    {
        "risk_area": "shortcut learning",
        "required_control": "group-based split, challenge set, and region-of-interest review",
        "evidence_to_keep": "split audit and error-analysis samples",
    },
    {
        "risk_area": "localized performance",
        "required_control": "slice metrics by relevant operational groups",
        "evidence_to_keep": "slice report and remediation record",
    },
    {
        "risk_area": "uncertain decisions",
        "required_control": "calibrated probabilities and human-review band",
        "evidence_to_keep": "threshold policy and review outcomes",
    },
])

display(governance_checklist_df)

vision_system_card = {
    "chapter": 15,
    "system_name": "synthetic_package_defect_triage",
    "decision": "route visibly defective product images for review or rework",
    "unit_of_analysis": "one cropped product image",
    "target": "visible defect present after the image is captured",
    "labeled_training_fraction": float(len(labeled_train_meta) / len(train_meta)),
    "split_strategy": "camera-group split: cameras 0-3 train, camera 4 validation, camera 5 test",
    "challenge_set": "new camera with dim, warm, and glare-heavy conditions",
    "model": "tiny CNN initialized from product-family pretraining and fine-tuned",
    "calibration": "Platt scaling fit on validation logits",
    "cost_threshold": float(cost_threshold),
    "triage_low_threshold": float(LOW_THRESHOLD),
    "triage_high_threshold": float(HIGH_THRESHOLD),
    "privacy_controls": [
        "crop unnecessary context",
        "restrict access to raw imagery",
        "retain derived metrics when raw images are not needed",
    ],
    "known_limitations": [
        "synthetic images are less complex than real operating scenes",
        "performance depends on defect visibility and capture quality",
        "new packaging and camera conditions require monitoring and revalidation",
    ],
}

print(json.dumps(vision_system_card, indent=2))

In [ ]:
# ============================================================
# 11.3 Save shareable analysis artifacts
# ============================================================
target_meta.to_csv(OUT_DIR / "ch15_target_image_metadata.csv", index=False)
model_results_df.to_csv(OUT_DIR / "ch15_model_results.csv", index=False)
calibration_summary.to_csv(OUT_DIR / "ch15_calibration_summary.csv", index=False)
slice_metrics_df.to_csv(OUT_DIR / "ch15_slice_metrics.csv", index=False)
spatial_metrics_df.to_csv(OUT_DIR / "ch15_spatial_metrics.csv", index=False)
document_results_df.to_csv(OUT_DIR / "ch15_document_field_results.csv", index=False)
retrieval_summary_df.to_csv(OUT_DIR / "ch15_retrieval_summary.csv", index=False)
monitoring_summary_df.to_csv(OUT_DIR / "ch15_monitoring_summary.csv", index=False)
np.save(OUT_DIR / "ch15_visual_embeddings.npy", visual_embeddings)
torch.save(fine_tuned_model.state_dict(), OUT_DIR / "ch15_fine_tuned_model_state.pt")

with open(OUT_DIR / "ch15_vision_system_card.json", "w", encoding="utf-8") as f:
    json.dump(vision_system_card, f, indent=2)

print("Saved artifacts:")
for path in sorted(OUT_DIR.iterdir()):
    print(" -", path)

## Decision guide

Use classification when one image-level decision is sufficient. Move to detection when the workflow needs locations or counts, and to segmentation when area or shape drives the decision. Use document vision when text roles depend on page layout. Use embeddings when the business question is about similarity, ranking, search, recommendation, clustering, or deduplication.

Start evaluation at the real deployment boundary. A camera-, store-, product-, user-, or time-group split is often more credible than a random image split. Begin with a transferred backbone and a simple head, then fine-tune only when error analysis shows that domain adaptation is necessary. Treat confidence as an operational variable by calibrating scores, assigning error costs, and reserving an uncertainty band for human review.

## Exercises

1. Change the camera assignments in the group split. What happens when a high-defect-rate camera becomes the test camera?

2. Train the target CNN on uncropped images by setting `crop_border=False`. Compare held-out-camera and challenge performance. Does the model become more sensitive to camera context?

3. Remove one augmentation from `augment_array`. Which slice changes most, normal, dim, warm, or glare?

4. Increase `FALSE_NEGATIVE_COST` from 8 to 20. How does the selected threshold, recall, and review workload change?

5. Change `TARGET_REVIEW_RATE` from 0.25 to 0.10 and 0.40. Quantify the trade-off between automated coverage and the precision of auto-routed cases.

6. Change the detection IoU threshold from 0.50 to 0.75. Why do precision and recall change even though the predicted boxes are unchanged?

7. Corrupt the `TOTAL` anchor instead of the amount token. Which document extraction rule fails, and what fallback would you add?

8. Redefine retrieval relevance using `style_id` instead of `product_family`. Does the current embedding still support the business objective?

9. Change `patch_size` from 8 to 4 or 16. Compare the number of tokens and the number of self-attention pair comparisons.

10. Add one monitoring rule that flags a batch when brightness, prediction rate, or review workload moves beyond a tolerance learned from the reference batch.

## Wrap-up

This notebook built a complete vision workflow around business decisions rather than around a single architecture. It covered image metadata, group-based splitting, shortcut detection, CNNs, transfer learning, calibration, thresholds, human review, slice evaluation, detection, segmentation, document fields, embeddings, self-supervised views, transformer patches, foundation model strategy, monitoring, and governance. The stable lesson is that useful vision systems turn pixels into measured, decision-relevant, and monitored outputs.